# Langkah 5b Anggota A — Skrip Reproduksi Fitur (`extract_features.py`)

**Tujuan:** memenuhi syarat *Definition of done* Anggota A di Rencana Kerja:

> `python extract_features.py --config config.json` dari nol menghasilkan `features.csv` yang identik.

Selama ini tabel fitur dibuat lewat dua notebook (langkah 4 dan 5). Skrip ini menggabungkan keduanya menjadi **satu perintah**, sehingga orang lain (dosen, anggota kelompok, atau pembaca repo GitHub) bisa membuat ulang tabel fitur dari audio asli tanpa membuka notebook.

**Alur skrip:** audio asli 48 kHz → preprocessing (`src/preprocess.py`) → 90 fitur dua versi (`src/features.py`) → `features_none.csv`, `features_peak.csv`, `feature_names.csv`, `features_meta.json`. Semua parameter dibaca dari `config.json`.

**Cara membuktikan "identik":** skrip dijalankan dari audio asli ke folder terpisah (`data/features_repro/`), lalu hasilnya dibandingkan sel per sel dengan tabel resmi di `data/features/`. Tabel resmi **tidak ditimpa**.

**Input dan output**

| Jenis | Lokasi (di `MyDrive/project07/`) | Keterangan |
| --- | --- | --- |
| Input | `config.json` | Bagian `preprocessing` dan `features` |
| Input | `src/preprocess.py`, `src/features.py` | Dari langkah 4 dan 5 |
| Input | `data/raw/ravdess_speech/` | Audio asli dari langkah 1 |
| Input | `data/features/` | Tabel resmi sebagai pembanding |
| Output | `extract_features.py` | Skrip reproduksi (di folder utama `project07/`) |
| Output | `data/features/repro_check.json` | Bukti perbandingan (identik atau tidak) |
| Output | `docs/README_extract_features.md` | Cara memakai skrip |
| Sementara | `data/features_repro_test/`, `data/features_repro/` | Hasil uji; dihapus di Sel 6 setelah terbukti identik |

**Prasyarat:** langkah 1, 4, dan 5 selesai. **Perkiraan waktu:** 15–30 menit (Sel 4 paling lama).

**Cara pakai:** jalankan sel berurutan dengan `Shift + Enter`.

## Sel 1 — Persiapan dan cek prasyarat

In [1]:
import os
import sys
import json
import shutil
import subprocess
from datetime import datetime

import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/project07'
except ImportError:
    BASE_DIR = os.path.abspath('project07')

CONFIG_PATH = os.path.join(BASE_DIR, 'config.json')
SCRIPT_PATH = os.path.join(BASE_DIR, 'extract_features.py')
FEAT_DIR = os.path.join(BASE_DIR, 'data', 'features')
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'ravdess_speech')
DOC_DIR = os.path.join(BASE_DIR, 'docs')
TEST_OUT = os.path.join(BASE_DIR, 'data', 'features_repro_test')
FULL_OUT = os.path.join(BASE_DIR, 'data', 'features_repro')
os.makedirs(DOC_DIR, exist_ok=True)

wajib = [CONFIG_PATH, os.path.join(BASE_DIR, 'src', 'preprocess.py'), os.path.join(BASE_DIR, 'src', 'features.py'),
         RAW_DIR, os.path.join(FEAT_DIR, 'features_none.csv'), os.path.join(FEAT_DIR, 'features_peak.csv'),
         os.path.join(FEAT_DIR, 'feature_names.csv')]
hilang = [p for p in wajib if not os.path.exists(p)]
if hilang:
    raise FileNotFoundError('Prasyarat tidak ditemukan:\n' + '\n'.join(hilang))

with open(CONFIG_PATH, encoding='utf-8') as f:
    CONFIG = json.load(f)
n_ref = len(pd.read_csv(os.path.join(FEAT_DIR, 'features_none.csv'), usecols=['file']))
N_CPU = os.cpu_count() or 1

def jalankan(argumen):
    # Jalankan skrip seperti di terminal, tampilkan keluarannya baris per baris, kembalikan exit code.
    cmd = [sys.executable, 'extract_features.py'] + argumen
    print('$ python', ' '.join(cmd[1:]))
    proses = subprocess.Popen(cmd, cwd=BASE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for baris in proses.stdout:
        print(baris, end='')
    proses.wait()
    print(f'\nexit code: {proses.returncode}')
    return proses.returncode

print('BASE_DIR     :', BASE_DIR)
print('Tabel resmi  :', n_ref, 'baris')
print('config_hash  : preprocessing', CONFIG['preprocessing']['config_hash'], '| features', CONFIG['features']['config_hash'])
print('CPU tersedia :', N_CPU)

Mounted at /content/drive
BASE_DIR     : /content/drive/MyDrive/project07
Tabel resmi  : 1440 baris
config_hash  : preprocessing d930ff28cd | features 0c3114fc78
CPU tersedia : 2


## Sel 2 — Tulis skrip `extract_features.py`

Baris pertama sel ini (`%%writefile`) menyimpan isi sel menjadi file `project07/extract_features.py`. Skrip ini yang nanti di-push ke GitHub.

Hal penting di dalam skrip:
- **Urutan file** sama dengan `metadata.csv` (diurutkan berdasarkan nama), dan kolom info (aktor, gender, emosi, intensitas) dibaca langsung dari nama file. Jadi skrip tidak bergantung pada `metadata.csv`.
- **Langkah per file** sama persis dengan langkah 4 dan 5: preprocessing tanpa normalisasi, F0 dihitung sekali, fitur versi `none`, lalu normalisasi puncak dan fitur versi `peak`.
- **config_hash** dihitung ulang dan dicocokkan dengan `config.json`, supaya ketahuan jika parameter berubah.
- **`--compare`** membandingkan hasil dengan tabel lain sel per sel dan menulis `compare_report.json`.
- **Exit code**: 0 = sukses (dan identik), 1 = ada file gagal atau hasil berbeda, 2 = masalah input.

In [2]:
%%writefile {SCRIPT_PATH}
#!/usr/bin/env python3
"""
extract_features.py — Membuat ulang tabel fitur Proyek 07 dari audio asli RAVDESS.

Penanggung jawab : Anggota A
Versi            : 1.0

Satu perintah ini menjalankan seluruh alur langkah 4 dan 5 dari nol:
  audio asli 48 kHz -> preprocessing (src/preprocess.py) -> 90 fitur (src/features.py)
  -> features_none.csv, features_peak.csv, feature_names.csv, features_meta.json

Semua parameter dibaca dari config.json (bagian "preprocessing" dan "features"),
sehingga hasilnya identik dengan tabel fitur resmi selama config dan versi library sama.

Contoh (dijalankan dari folder project07/):
  python extract_features.py --config config.json
  python extract_features.py --config config.json --compare data/features
  python extract_features.py --config config.json --actors 1 2 --out-dir data/features_repro_test

Kode keluar (exit code): 0 = sukses (dan identik jika --compare dipakai),
1 = ada file gagal atau hasil tidak identik, 2 = masalah input/konfigurasi.
"""
import os
import sys
import json
import glob
import time
import hashlib
import argparse
import platform
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

SCRIPT_VERSION = '1.0'
INFO_COLS = ['file', 'actor', 'gender', 'emotion', 'intensity']
EMOTIONS = {1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad',
            5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'}
INTENSITIES = {1: 'normal', 2: 'strong'}
PRE_KEYS = ['target_sr', 'mono', 'trim_top_db', 'trim_frame_length', 'trim_hop_length', 'peak_target']
FEAT_KEYS = ['sr', 'n_fft', 'hop_length', 'n_mfcc', 'n_mels', 'delta_width', 'f0_fmin', 'f0_fmax',
             'f0_frame_length', 'rolloff_percent', 'contrast_fmin', 'contrast_n_bands']

# Diisi di setiap proses pekerja oleh _init_worker
_PP = _FT = _PRE = _FEAT = _RAW = None


def parse_args(argv=None):
    p = argparse.ArgumentParser(description='Buat ulang tabel fitur Proyek 07 dari audio asli RAVDESS.')
    p.add_argument('--config', default='config.json', help='Lokasi config.json (default: config.json)')
    p.add_argument('--raw-dir', default=None,
                   help='Folder audio asli berisi Actor_XX/ (default: data/raw/ravdess_speech di sebelah config)')
    p.add_argument('--out-dir', default=None,
                   help='Folder output (default: data/features_repro). Tidak menimpa data/features kecuali diminta.')
    p.add_argument('--actors', type=int, nargs='+', default=None,
                   help='Hanya proses aktor tertentu, misalnya --actors 1 2 (untuk uji cepat)')
    p.add_argument('--compare', default=None,
                   help='Folder tabel fitur pembanding, misalnya data/features. Hasil dibandingkan sel per sel.')
    p.add_argument('--n-jobs', type=int, default=1, help='Jumlah proses paralel (default 1)')
    p.add_argument('--rtol', type=float, default=1e-6, help='Toleransi relatif saat membandingkan (default 1e-6)')
    p.add_argument('--atol', type=float, default=1e-9, help='Toleransi absolut saat membandingkan (default 1e-9)')
    return p.parse_args(argv)


def config_hashes(pre_cfg, feat_cfg):
    """Hitung config_hash persis seperti notebook langkah 4 dan 5."""
    pre_hash = hashlib.md5(json.dumps(pre_cfg, sort_keys=True).encode()).hexdigest()[:10]
    feat_hash = hashlib.md5(json.dumps({**feat_cfg, 'pre': pre_hash}, sort_keys=True).encode()).hexdigest()[:10]
    return pre_hash, feat_hash


def parse_filename(rel_path):
    """Actor_01/03-01-05-02-01-01-01.wav -> kolom info."""
    parts = os.path.splitext(os.path.basename(rel_path))[0].split('-')
    if len(parts) != 7:
        raise ValueError(f'Nama file tidak sesuai format RAVDESS: {rel_path}')
    emo, inten, actor = int(parts[2]), int(parts[3]), int(parts[6])
    return {'file': rel_path.replace(os.sep, '/'), 'actor': actor,
            'gender': 'male' if actor % 2 == 1 else 'female',
            'emotion': EMOTIONS[emo], 'intensity': INTENSITIES[inten]}


def list_files(raw_dir, actors=None):
    """Daftar file dengan urutan sama seperti metadata.csv (langkah 1: sorted glob)."""
    paths = sorted(glob.glob(os.path.join(raw_dir, '**', '*.wav'), recursive=True))
    rows = [parse_filename(os.path.relpath(p, raw_dir)) for p in paths]
    if actors:
        rows = [r for r in rows if r['actor'] in set(actors)]
    return pd.DataFrame(rows, columns=INFO_COLS)


def _init_worker(src_dir, pre_cfg, feat_cfg, raw_dir):
    global _PP, _FT, _PRE, _FEAT, _RAW
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import preprocess as pp
    import features as ft
    _PP, _FT, _PRE, _FEAT, _RAW = pp, ft, pre_cfg, feat_cfg, raw_dir


def _process_one(rel_path):
    """Satu file: preprocessing (langkah 4) lalu fitur dua versi (langkah 5)."""
    try:
        y, sr, _ = _PP.preprocess_file(os.path.join(_RAW, rel_path), _PRE, 'none')
        # Langkah 4 menyimpan audio sebagai WAV float32 dan langkah 5 membacanya kembali;
        # float32 tersimpan tanpa kehilangan, jadi cukup dipastikan bertipe float32.
        y = np.asarray(y, dtype=np.float32)
        f0 = _FT.extract_f0_features(y, _FEAT)
        x_none = _FT.extract_features(y, _FEAT, f0_feats=f0)
        y_peak = _PP.normalize(y, 'peak', _PRE['peak_target'])
        x_peak = _FT.extract_features(y_peak, _FEAT, f0_feats=f0)
        return rel_path, x_none, x_peak, None
    except Exception as e:  # dilaporkan, tidak menghentikan proses
        return rel_path, None, None, f'{type(e).__name__}: {e}'


def compare_tables(new_path, ref_path, rtol, atol):
    """Bandingkan dua tabel fitur. Kembalikan dict ringkasan."""
    new = pd.read_csv(new_path, float_precision='round_trip')
    ref = pd.read_csv(ref_path, float_precision='round_trip')
    rep = {'file_baru': new_path, 'file_pembanding': ref_path,
           'baris_baru': len(new), 'baris_pembanding': len(ref)}
    rep['kolom_sama'] = list(new.columns) == list(ref.columns)
    ref = ref[ref['file'].isin(new['file'])]           # mendukung --actors
    rep['baris_dibandingkan'] = len(ref)
    rep['urutan_file_sama'] = list(new['file']) == list(ref['file'])
    ref = ref.set_index('file').loc[new['file']].reset_index()
    info = [c for c in INFO_COLS if c != 'file']
    rep['info_sama'] = bool((new[info].astype(str).values == ref[info].astype(str).values).all())
    fcols = [c for c in new.columns if c not in INFO_COLS]
    a = new[fcols].to_numpy(dtype=float)
    b = ref[fcols].to_numpy(dtype=float)
    diff = np.abs(a - b)
    rep['jumlah_sel'] = int(a.size)
    rep['sel_identik_persis'] = int((a == b).sum())
    rep['selisih_abs_maks'] = float(diff.max()) if diff.size else 0.0
    rel = diff / np.maximum(np.abs(b), 1e-12)
    rep['selisih_rel_maks'] = float(rel.max()) if rel.size else 0.0
    close = np.isclose(a, b, rtol=rtol, atol=atol)
    rep['sel_dalam_toleransi'] = int(close.sum())
    worst = np.argsort(-(diff.max(axis=0)))[:3] if diff.size else []
    rep['fitur_selisih_terbesar'] = {fcols[i]: float(diff[:, i].max()) for i in worst}
    rep['identik'] = bool(rep['kolom_sama'] and rep['info_sama'] and close.all()
                          and rep['baris_dibandingkan'] == len(new))
    return rep


def main(argv=None):
    args = parse_args(argv)
    t0 = time.time()
    cfg_path = os.path.abspath(args.config)
    if not os.path.exists(cfg_path):
        print(f'[ERROR] config tidak ditemukan: {cfg_path}')
        return 2
    base = os.path.dirname(cfg_path)
    with open(cfg_path, encoding='utf-8') as f:
        config = json.load(f)
    for sec in ('preprocessing', 'features'):
        if sec not in config:
            print(f'[ERROR] config.json tidak punya bagian "{sec}". Jalankan notebook langkah 4 dan 5 dulu.')
            return 2

    pre_cfg = {k: config['preprocessing'][k] for k in PRE_KEYS}
    feat_cfg = {k: config['features'][k] for k in FEAT_KEYS}
    pre_hash, feat_hash = config_hashes(pre_cfg, feat_cfg)
    src_dir = os.path.join(base, 'src')
    raw_dir = os.path.abspath(args.raw_dir or os.path.join(base, 'data', 'raw', 'ravdess_speech'))
    out_dir = os.path.abspath(args.out_dir or os.path.join(base, 'data', 'features_repro'))

    for p, nama in [(os.path.join(src_dir, 'preprocess.py'), 'src/preprocess.py'),
                    (os.path.join(src_dir, 'features.py'), 'src/features.py'), (raw_dir, 'folder audio asli')]:
        if not os.path.exists(p):
            print(f'[ERROR] {nama} tidak ditemukan: {p}')
            return 2

    print(f'extract_features.py v{SCRIPT_VERSION}')
    print(f'  config     : {cfg_path}')
    print(f'  audio asli : {raw_dir}')
    print(f'  output     : {out_dir}')
    for nama, h_cfg, h_hit in [('preprocessing', config['preprocessing'].get('config_hash'), pre_hash),
                               ('features', config['features'].get('config_hash'), feat_hash)]:
        tanda = 'cocok' if h_cfg == h_hit else f'BERBEDA dari config.json ({h_cfg})'
        print(f'  config_hash {nama:<13}: {h_hit} ({tanda})')

    files = list_files(raw_dir, args.actors)
    if files.empty:
        print('[ERROR] tidak ada file WAV yang ditemukan.')
        return 2
    print(f'  file       : {len(files)}' + (f' (aktor {args.actors})' if args.actors else ''))

    _init_worker(src_dir, pre_cfg, feat_cfg, raw_dir)
    ft = _FT
    hasil, gagal = {}, {}
    n = len(files)
    if args.n_jobs > 1:
        with ProcessPoolExecutor(max_workers=args.n_jobs, initializer=_init_worker,
                                 initargs=(src_dir, pre_cfg, feat_cfg, raw_dir)) as ex:
            it = ex.map(_process_one, files['file'], chunksize=8)
            for i, (f, xn, xp, err) in enumerate(it, start=1):
                (gagal.__setitem__(f, err) if err else hasil.__setitem__(f, (xn, xp)))
                if i % 100 == 0 or i == n:
                    print(f'  {i}/{n} file | {(time.time() - t0) / 60:.1f} menit', flush=True)
    else:
        for i, f in enumerate(files['file'], start=1):
            f, xn, xp, err = _process_one(f)
            (gagal.__setitem__(f, err) if err else hasil.__setitem__(f, (xn, xp)))
            if i % 100 == 0 or i == n:
                print(f'  {i}/{n} file | {(time.time() - t0) / 60:.1f} menit', flush=True)

    os.makedirs(out_dir, exist_ok=True)
    ok = files[files['file'].isin(hasil)].reset_index(drop=True)
    paths = {}
    for k, versi in enumerate(['none', 'peak']):
        X = pd.DataFrame([hasil[f][k] for f in ok['file']], columns=ft.FEATURE_NAMES)
        paths[versi] = os.path.join(out_dir, f'features_{versi}.csv')
        pd.concat([ok, X], axis=1).to_csv(paths[versi], index=False)
    pd.DataFrame([{'index': i, 'name': nm, 'group': g, 'description': d, 'scale_dependent': s}
                  for i, (nm, g, d, s) in enumerate(ft.FEATURE_TABLE)]).to_csv(
        os.path.join(out_dir, 'feature_names.csv'), index=False)

    import librosa
    import soundfile
    meta = {'config_hash': feat_hash, 'preprocessing_hash': pre_hash,
            'created': datetime.now().strftime('%Y-%m-%d %H:%M'), 'n_files': len(ok), 'n_failed': len(gagal),
            'actors': args.actors or 'semua', 'script_version': SCRIPT_VERSION,
            'library': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                        'librosa': librosa.__version__, 'soundfile': soundfile.__version__},
            'failed': gagal}
    with open(os.path.join(out_dir, 'features_meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)
    print(f'Tersimpan di {out_dir}: features_none.csv, features_peak.csv, feature_names.csv, features_meta.json')
    print(f'Berhasil {len(ok)} file, gagal {len(gagal)} file, waktu {(time.time() - t0) / 60:.1f} menit')
    for f, e in list(gagal.items())[:10]:
        print(f'  GAGAL {f}: {e}')

    kode = 1 if gagal else 0
    if args.compare:
        ref_dir = os.path.abspath(args.compare)
        laporan = {'rtol': args.rtol, 'atol': args.atol}
        for versi in ['none', 'peak']:
            ref = os.path.join(ref_dir, f'features_{versi}.csv')
            if not os.path.exists(ref):
                print(f'[ERROR] pembanding tidak ditemukan: {ref}')
                return 2
            r = compare_tables(paths[versi], ref, args.rtol, args.atol)
            laporan[versi] = r
            print(f"Perbandingan {versi}: {'IDENTIK' if r['identik'] else 'BERBEDA'} | "
                  f"{r['sel_identik_persis']}/{r['jumlah_sel']} sel sama persis | "
                  f"selisih abs maks {r['selisih_abs_maks']:.3g} | rel maks {r['selisih_rel_maks']:.3g}")
            if not r['identik']:
                kode = 1
        ref_names = os.path.join(ref_dir, 'feature_names.csv')
        if os.path.exists(ref_names):
            same = pd.read_csv(ref_names).equals(pd.read_csv(os.path.join(out_dir, 'feature_names.csv')))
            laporan['feature_names_sama'] = bool(same)
            print(f"feature_names.csv: {'sama' if same else 'BERBEDA'}")
            if not same:
                kode = 1
        with open(os.path.join(out_dir, 'compare_report.json'), 'w', encoding='utf-8') as f:
            json.dump(laporan, f, indent=2)
        print('Laporan perbandingan:', os.path.join(out_dir, 'compare_report.json'))
    return kode


if __name__ == '__main__':
    sys.exit(main())


Writing /content/drive/MyDrive/project07/extract_features.py


In [3]:
kode = jalankan(['--help'])

$ python extract_features.py --help
usage: extract_features.py [-h] [--config CONFIG] [--raw-dir RAW_DIR]
                           [--out-dir OUT_DIR] [--actors ACTORS [ACTORS ...]]
                           [--compare COMPARE] [--n-jobs N_JOBS] [--rtol RTOL]
                           [--atol ATOL]

Buat ulang tabel fitur Proyek 07 dari audio asli RAVDESS.

options:
  -h, --help            show this help message and exit
  --config CONFIG       Lokasi config.json (default: config.json)
  --raw-dir RAW_DIR     Folder audio asli berisi Actor_XX/ (default:
                        data/raw/ravdess_speech di sebelah config)
  --out-dir OUT_DIR     Folder output (default: data/features_repro). Tidak
                        menimpa data/features kecuali diminta.
  --actors ACTORS [ACTORS ...]
                        Hanya proses aktor tertentu, misalnya --actors 1 2
                        (untuk uji cepat)
  --compare COMPARE     Folder tabel fitur pembanding, misalnya data/features.
   

## Sel 3 — Uji cepat (aktor 1 dan 2, ±1–2 menit)

Sebelum menjalankan 1.440 file, skrip dicoba pada 120 file milik aktor 1 dan 2. Hasilnya harus **IDENTIK** dengan baris yang sama di tabel resmi. Jika muncul **BERBEDA**, hentikan dan kirim keluarannya ke saya.

In [4]:
kode_uji = jalankan(['--config', 'config.json', '--actors', '1', '2',
                     '--out-dir', 'data/features_repro_test', '--compare', 'data/features',
                     '--n-jobs', str(N_CPU)])
if kode_uji != 0:
    print('\n[CEK] Uji cepat tidak identik atau gagal. Jangan lanjut ke Sel 4; kirim keluaran di atas.')

$ python extract_features.py --config config.json --actors 1 2 --out-dir data/features_repro_test --compare data/features --n-jobs 2
extract_features.py v1.0
  config     : /content/drive/MyDrive/project07/config.json
  audio asli : /content/drive/MyDrive/project07/data/raw/ravdess_speech
  output     : /content/drive/MyDrive/project07/data/features_repro_test
  config_hash preprocessing: d930ff28cd (cocok)
  config_hash features     : 0c3114fc78 (cocok)
  file       : 120 (aktor [1, 2])
  100/120 file | 2.5 menit
  120/120 file | 2.7 menit
Tersimpan di /content/drive/MyDrive/project07/data/features_repro_test: features_none.csv, features_peak.csv, feature_names.csv, features_meta.json
Berhasil 120 file, gagal 0 file, waktu 2.7 menit
Perbandingan none: IDENTIK | 10800/10800 sel sama persis | selisih abs maks 0 | rel maks 0
Perbandingan peak: IDENTIK | 10800/10800 sel sama persis | selisih abs maks 0 | rel maks 0
feature_names.csv: sama
Laporan perbandingan: /content/drive/MyDrive/proje

## Sel 4 — Reproduksi penuh dari nol (1.440 file, ±10–25 menit)

Ini bukti utama *Definition of done*: seluruh tabel fitur dibuat ulang dari audio asli, lalu dibandingkan dengan tabel resmi. Pemrosesan memakai semua CPU Colab sekaligus (`--n-jobs`), dan urutan hasil tetap sama.

Jika Colab terputus di tengah jalan, jalankan ulang Sel 1, 2, dan sel ini. Skrip tidak memakai checkpoint; ia akan mengulang dari awal.

In [5]:
kode_penuh = jalankan(['--config', 'config.json', '--out-dir', 'data/features_repro',
                       '--compare', 'data/features', '--n-jobs', str(N_CPU)])

$ python extract_features.py --config config.json --out-dir data/features_repro --compare data/features --n-jobs 2
extract_features.py v1.0
  config     : /content/drive/MyDrive/project07/config.json
  audio asli : /content/drive/MyDrive/project07/data/raw/ravdess_speech
  output     : /content/drive/MyDrive/project07/data/features_repro
  config_hash preprocessing: d930ff28cd (cocok)
  config_hash features     : 0c3114fc78 (cocok)
  file       : 1440
  100/1440 file | 1.1 menit
  200/1440 file | 2.4 menit
  300/1440 file | 3.8 menit
  400/1440 file | 5.2 menit
  500/1440 file | 6.4 menit
  600/1440 file | 7.7 menit
  700/1440 file | 9.1 menit
  800/1440 file | 10.2 menit
  900/1440 file | 11.5 menit
  1000/1440 file | 12.7 menit
  1100/1440 file | 14.1 menit
  1200/1440 file | 15.4 menit
  1300/1440 file | 16.8 menit
  1400/1440 file | 18.0 menit
  1440/1440 file | 18.6 menit
Tersimpan di /content/drive/MyDrive/project07/data/features_repro: features_none.csv, features_peak.csv, featu

## Sel 5 — Verifikasi

Sel ini membaca laporan perbandingan dari Sel 4. Setiap pemeriksaan harus bertanda **OK**.

In [6]:
with open(os.path.join(FULL_OUT, 'compare_report.json'), encoding='utf-8') as f:
    REP = json.load(f)
with open(os.path.join(FULL_OUT, 'features_meta.json'), encoding='utf-8') as f:
    META = json.load(f)

laporan = []
def cek(nama, kondisi, detail=''):
    laporan.append(f"[{'OK' if kondisi else 'CEK'}] {nama}" + (f' — {detail}' if detail else ''))

cek('Skrip selesai dengan exit code 0', kode_penuh == 0, f'exit code {kode_penuh}')
cek('Tidak ada file gagal', META['n_failed'] == 0, f"{META['n_files']} berhasil, {META['n_failed']} gagal")
cek('Jumlah baris sama dengan tabel resmi', REP['none']['baris_baru'] == n_ref == REP['peak']['baris_baru'],
    f"{REP['none']['baris_baru']} vs {n_ref}")
cek('config_hash hasil = config.json', META['config_hash'] == CONFIG['features']['config_hash']
    and META['preprocessing_hash'] == CONFIG['preprocessing']['config_hash'],
    f"features {META['config_hash']}, preprocessing {META['preprocessing_hash']}")
for v in ['none', 'peak']:
    r = REP[v]
    cek(f'features_{v}.csv: kolom, urutan file, dan kolom info sama',
        r['kolom_sama'] and r['urutan_file_sama'] and r['info_sama'])
    cek(f'features_{v}.csv: semua nilai fitur identik', r['identik'],
        f"{r['sel_identik_persis']}/{r['jumlah_sel']} sel sama persis, selisih maks {r['selisih_abs_maks']:.3g}")
cek('feature_names.csv sama', REP.get('feature_names_sama', False))

for t in laporan:
    print(t)
n_ok = sum(t.startswith('[OK') for t in laporan)
print(f'\n{n_ok}/{len(laporan)} OK')
persis = all(REP[v]['sel_identik_persis'] == REP[v]['jumlah_sel'] for v in ['none', 'peak'])
print('Kesimpulan:', 'IDENTIK PERSIS (bit per bit)' if persis else
      ('IDENTIK dalam toleransi pembulatan' if all(REP[v]['identik'] for v in ['none', 'peak']) else 'TIDAK IDENTIK'))
print('Versi library:', META['library'])

[OK] Skrip selesai dengan exit code 0 — exit code 0
[OK] Tidak ada file gagal — 1440 berhasil, 0 gagal
[OK] Jumlah baris sama dengan tabel resmi — 1440 vs 1440
[OK] config_hash hasil = config.json — features 0c3114fc78, preprocessing d930ff28cd
[OK] features_none.csv: kolom, urutan file, dan kolom info sama
[OK] features_none.csv: semua nilai fitur identik — 129600/129600 sel sama persis, selisih maks 0
[OK] features_peak.csv: kolom, urutan file, dan kolom info sama
[OK] features_peak.csv: semua nilai fitur identik — 129600/129600 sel sama persis, selisih maks 0
[OK] feature_names.csv sama

9/9 OK
Kesimpulan: IDENTIK PERSIS (bit per bit)
Versi library: {'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3', 'librosa': '0.11.0', 'soundfile': '0.14.0'}


**Jika ada CEK:** perbedaan kecil (selisih sangat kecil di banyak sel) biasanya muncul karena versi library Colab berubah sejak langkah 5 dijalankan. Kirim keluaran Sel 4 dan Sel 5 ke saya; jangan menimpa tabel resmi.

## Sel 6 — Simpan bukti dan bersihkan hasil sementara

Bukti perbandingan disimpan ke `data/features/repro_check.json` (kecil, ikut di-push ke GitHub). Folder hasil uji dihapus karena isinya sama dengan tabel resmi, kecuali hasilnya tidak identik (folder dibiarkan untuk diperiksa).

In [7]:
bukti = {'tanggal': datetime.now().strftime('%Y-%m-%d %H:%M'),
         'perintah': 'python extract_features.py --config config.json --compare data/features',
         'exit_code': kode_penuh, 'verifikasi': laporan,
         'meta': {k: META[k] for k in ['config_hash', 'preprocessing_hash', 'n_files', 'n_failed', 'script_version', 'library']},
         'perbandingan': REP}
with open(os.path.join(FEAT_DIR, 'repro_check.json'), 'w', encoding='utf-8') as f:
    json.dump(bukti, f, indent=2)
print('Tersimpan:', os.path.join(FEAT_DIR, 'repro_check.json'))

semua_ok = n_ok == len(laporan)
for d in [TEST_OUT, FULL_OUT]:
    if os.path.exists(d):
        if semua_ok:
            shutil.rmtree(d)
            print('Dihapus:', d)
        else:
            print('Dibiarkan untuk diperiksa:', d)

Tersimpan: /content/drive/MyDrive/project07/data/features/repro_check.json
Dihapus: /content/drive/MyDrive/project07/data/features_repro_test
Dihapus: /content/drive/MyDrive/project07/data/features_repro


## Sel 7 — Dokumentasi standar

Sel ini menyimpan `docs/README_extract_features.md`.

In [8]:
tgl = datetime.now().strftime('%Y-%m-%d')
lib = META['library']
readme = f'''# extract_features.py — Reproduksi Tabel Fitur

Penanggung jawab: Anggota A · Versi skrip: {META['script_version']} · Diverifikasi: {tgl} · config_hash fitur: `{META['config_hash']}` (preprocessing `{META['preprocessing_hash']}`)

## Tujuan

Membuat ulang tabel fitur Proyek 07 dari audio asli RAVDESS dengan satu perintah, sesuai syarat Definition of done Anggota A. Skrip menjalankan alur langkah 4 (preprocessing) dan langkah 5 (ekstraksi 90 fitur, dua versi normalisasi) dengan parameter dari `config.json`.

## Prasyarat

- Audio asli di `data/raw/ravdess_speech/Actor_XX/*.wav` (cara mengunduh: `data/README.md` atau notebook langkah 1)
- `config.json`, `src/preprocess.py`, `src/features.py`
- Library: numpy, pandas, librosa, soundfile (versi yang dipakai saat verifikasi: Python {lib['python']}, numpy {lib['numpy']}, pandas {lib['pandas']}, librosa {lib['librosa']}, soundfile {lib['soundfile']})

## Cara memakai

Jalankan dari folder `project07/`:

```bash
# buat ulang semua tabel fitur (output ke data/features_repro/)
python extract_features.py --config config.json

# buat ulang dan bandingkan dengan tabel resmi
python extract_features.py --config config.json --compare data/features

# uji cepat untuk aktor tertentu, paralel 2 proses
python extract_features.py --config config.json --actors 1 2 --n-jobs 2 --out-dir data/features_repro_test --compare data/features
```

| Argumen | Default | Keterangan |
| --- | --- | --- |
| `--config` | `config.json` | Lokasi config; folder config dianggap folder utama proyek |
| `--raw-dir` | `data/raw/ravdess_speech` | Folder audio asli |
| `--out-dir` | `data/features_repro` | Folder output; tabel resmi tidak ditimpa |
| `--actors` | semua | Hanya aktor tertentu |
| `--compare` | – | Folder tabel pembanding; menulis `compare_report.json` |
| `--n-jobs` | 1 | Jumlah proses paralel; hasil tetap sama |
| `--rtol`, `--atol` | 1e-6, 1e-9 | Toleransi perbandingan |

Exit code: 0 = sukses (dan identik jika `--compare` dipakai), 1 = ada file gagal atau hasil berbeda, 2 = masalah input.

## Output

| File | Isi |
| --- | --- |
| `features_none.csv`, `features_peak.csv` | `file, actor, gender, emotion, intensity` + 90 fitur |
| `feature_names.csv` | `index, name, group, description, scale_dependent` |
| `features_meta.json` | config_hash, jumlah file, versi library, daftar file gagal |
| `compare_report.json` | Hasil perbandingan (hanya jika `--compare` dipakai) |

## Hasil verifikasi

Perintah `python extract_features.py --config config.json --compare data/features` dijalankan dari audio asli pada {tgl}.

''' + '\n'.join(f'- {t}' for t in laporan) + f'''

Bukti lengkap: `data/features/repro_check.json`.

## Catatan

- Hasil identik bit per bit hanya dijamin dengan versi library yang sama. Versi lain (terutama librosa dan numpy) dapat menghasilkan selisih pembulatan sangat kecil; gunakan `--compare` untuk memeriksanya.
- Kolom info dibaca dari nama file RAVDESS (`modality-channel-emotion-intensity-statement-repetition-actor`), dan urutan baris mengikuti urutan nama file, sama seperti `metadata.csv`.

## Riwayat versi

| Versi | Tanggal | Perubahan |
| --- | --- | --- |
| 1.0 | {tgl} | Versi pertama |
'''
with open(os.path.join(DOC_DIR, 'README_extract_features.md'), 'w', encoding='utf-8') as f:
    f.write(readme)
print('Tersimpan:', os.path.join(DOC_DIR, 'README_extract_features.md'))

Tersimpan: /content/drive/MyDrive/project07/docs/README_extract_features.md


## Selesai — yang dilakukan setelah notebook ini

1. Pastikan Sel 5 menampilkan semua **OK** dan kesimpulan **IDENTIK**.
2. Simpan notebook ke `project07/notebooks/Langkah5b_AnggotaA_Skrip_Reproduksi.ipynb`.
3. Tambahkan baris `data/features_repro*/` ke `.gitignore` untuk berjaga-jaga.
4. Kirim ke saya untuk Log Kerja: keluaran Sel 4 (beberapa baris terakhir) dan Sel 5.